# Presentation plots

Generates every plot in the project presentation **except** the "Final notebook" slides
(control vs campaign, voter animation), which come from `final_simulation.ipynb`.

| Slide | Plot | Section |
|---|---|---|
| 9 – From opinion to vote | Baseline support over 30 days (social influence only) | 1 |
| 14 – News and events | Support / opinion response to an event, with decay | 2 |
| 15 – Turnout boost | Turnout probability vs exposure; turnout with vs without mobilisation | 3 |
| 16 – Public polls and strategic voting | Sincere vs strategic table (C01, seed 42, poll on day 25) | 4 |
| 17 – Regional simulation | Seats / vote share vs spillover (0, 0.05, 0.1, 0.2) | 5 |
| 18 – Experiments | Vote-share change per scenario (mean ± 5–95% band over 30 seeds) | 5 |

Every figure is also saved to `figures/` (PNG + PDF) so it can be dropped into the beamer deck.
All data is synthetic. Run from the repository root. Sections 1–4 take seconds; section 5
runs the 30-seed experiment matrix and spillover sweep and takes several minutes
(lower `SEED_COUNT` for a quick preview).

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "election_sim").is_dir():
    raise RuntimeError("Run this notebook from the repository root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from constituency_loader import load_constituency
from election_sim import SimulationConfig, simulate
from election_sim.campaign import load_actions
from election_sim.campaign.models import CampaignAction
from election_sim.region import load_region
from election_sim.step5_cli import run_step5
from election_sim.voting import mobilized_propensity

DATA_DIR = ROOT / "data" / "synthetic_election_dataset"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)
SEED, DAYS, CONSTITUENCY_ID = 42, 30, 1

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})
CAND_COLORS = {"CAND_0001": "#2563eb", "CAND_0002": "#16a34a", "CAND_0003": "#e11d48"}
PARTY_COLORS = {"P1": "#6f789d", "P2": "#bd8b4e", "P3": "#4f8e89",
                "P4": "#987184", "P5": "#78916f", "IND": "#879195"}

def save(fig, name):
    """Save a figure as PNG and PDF for the slides."""
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=200)

data = load_constituency(DATA_DIR, CONSTITUENCY_ID, election_duration_days=DAYS)
no_events = {**data, "events": data["events"].iloc[:0]}
candidate_ids = sorted(data["candidates"].candidate_id)
print("C01 voters:", len(data["voters"]), "| candidates:", candidate_ids)

## 1. Baseline support over 30 days (slide 9)

Social influence only: no campaigns, no events. The frozen run (no social influence either)
is drawn dashed as a reference, so any movement in the solid lines comes from
bounded-confidence averaging alone.

In [ ]:
social_cfg = SimulationConfig(constituency_id=CONSTITUENCY_ID, days=DAYS,
                              lambda_campaign=0.0, lambda_event=0.0)
frozen_cfg = replace(social_cfg, lambda_social=0.0)
social = simulate(social_cfg, no_events, SEED)
frozen = simulate(frozen_cfg, no_events, SEED)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
for cid, g in social.daily_support.groupby("candidate_id"):
    axes[0].plot(g.day, g.mean_support, color=CAND_COLORS[cid], lw=2, label=cid)
for cid, g in frozen.daily_support.groupby("candidate_id"):
    axes[0].plot(g.day, g.mean_support, "--", color=CAND_COLORS[cid], lw=1, alpha=0.6)
axes[0].plot([], [], "k--", lw=1, alpha=0.6, label="frozen (reference)")
axes[0].set(title="Mean candidate support, C01 (social influence only)",
            xlabel="Day", ylabel="Mean support", xlim=(0, DAYS))
axes[0].legend(fontsize=8)

summary = social.daily_summary
axes[1].fill_between(summary.day, summary.q05, summary.q95, color="#64748b", alpha=0.15, label="5–95%")
axes[1].fill_between(summary.day, summary.q25, summary.q75, color="#64748b", alpha=0.3, label="25–75%")
axes[1].plot(summary.day, summary.q50, color="#0f172a", lw=2, label="median")
axes[1].set(title="Voter opinion distribution", xlabel="Day", ylabel="Opinion", xlim=(0, DAYS))
axes[1].legend(fontsize=8)
save(fig, "slide09_baseline_support")
plt.show()
print(social.candidate_results.round(3).to_string(index=False))

## 2. Response to an event, with decay (slide 14)

One hand-made event is added to an otherwise event-free run, and each run is compared with
the matched no-event run (same seed, so the difference is the event alone).

* **Left:** a negative story about CAND_0001 (`candidate_effect = -1`), days 8–10. The push
  accumulates during the active days, then fades with half-life *h*. `h = 0` means it never fades.
* **Right:** a neutral directional shock (`opinion_direction = +1`) on the same days moves
  opinions instead of a candidate's appeal; social influence then carries and dilutes it.

`lambda_event = 0.6` is a demonstration value (model default 0.2) so the shape is easy to see.

In [ ]:
EVENT_DAY, EVENT_DURATION = 8, 3
event_cfg = replace(social_cfg, lambda_event=0.6)

def one_event(**fields):
    row = dict(event_id="DEMO_EVENT", region_id=1, constituency_id=CONSTITUENCY_ID,
               event_day=EVENT_DAY, duration=EVENT_DURATION, intensity=1.0, credibility=1.0,
               target_group="General", issue="general", candidate_id=None,
               candidate_effect=None, source_type="News", event_type="Other")
    row.update(fields)
    return {**data, "events": pd.DataFrame([row])}

control = simulate(event_cfg, no_events, SEED)  # no events -> lambda_event has nothing to act on
control_support = control.daily_support.pivot(index="day", columns="candidate_id", values="mean_support")

scandal = one_event(candidate_id="CAND_0001", candidate_effect=-1.0, event_type="Scandal")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
for half_life, color in [(3, "#f59e0b"), (7, "#dc2626"), (0, "#475569")]:
    run = simulate(replace(event_cfg, event_memory_half_life_days=float(half_life)), scandal, SEED)
    support = run.daily_support.pivot(index="day", columns="candidate_id", values="mean_support")
    diff = (support - control_support)["CAND_0001"] * 100
    label = "no decay (h = 0)" if half_life == 0 else f"half-life h = {half_life} days"
    axes[0].plot(diff.index, diff.values, color=color, lw=2, label=label)
axes[0].axvspan(EVENT_DAY, EVENT_DAY + EVENT_DURATION - 1, color="#f59e0b", alpha=0.12, label="event active")
axes[0].axhline(0, color="k", lw=0.8)
axes[0].set(title="Negative story about CAND_0001", xlabel="Day",
            ylabel="Change in CAND_0001 support (pp)", xlim=(0, DAYS))
axes[0].legend(fontsize=8)

shock = one_event(shock_mode="directional", opinion_direction=1.0)
for half_life, color in [(3, "#f59e0b"), (7, "#dc2626"), (0, "#475569")]:
    run = simulate(replace(event_cfg, event_memory_half_life_days=float(half_life)), shock, SEED)
    diff = run.daily_summary.opinion_mean - control.daily_summary.opinion_mean
    label = "no decay (h = 0)" if half_life == 0 else f"h = {half_life} days"
    axes[1].plot(run.daily_summary.day, diff, color=color, lw=2, label=label)
axes[1].axvspan(EVENT_DAY, EVENT_DAY + EVENT_DURATION - 1, color="#f59e0b", alpha=0.12)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set(title="Neutral directional shock", xlabel="Day",
            ylabel="Change in mean opinion", xlim=(0, DAYS))
axes[1].legend(fontsize=8)
save(fig, "slide14_event_response")
plt.show()

## 3. Turnout boost (slide 15)

**Left:** the formula itself, $P(\text{vote}) = \sigma(\operatorname{logit}(p) + \gamma_{mob} E)$, for
voters with different baseline propensities. The boost is largest for undecided voters (p ≈ 0.5).

**Right:** a uniform ordinary campaign reaching 25% of C01 voters on days 10–24, run with
γ_mob = 0 (original turnout) and γ_mob > 0. Bars are the mean over 30 matched seeds with the
5–95% band. The campaign and seeds are identical; only γ_mob changes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
exposure = np.arange(0, 31)
for p, color in [(0.2, "#94a3b8"), (0.5, "#2563eb"), (0.8, "#0f172a")]:
    prob = mobilized_propensity(np.full(exposure.shape, p), exposure, np.zeros_like(exposure), gamma_mob=0.05)
    axes[0].plot(exposure, prob, color=color, lw=2, label=f"baseline propensity {p}")
    axes[0].axhline(p, color=color, lw=0.8, ls=":")
axes[0].set(title=r"Turnout probability vs exposure ($\gamma_{mob}=0.05$)",
            xlabel="Days reached by a campaign (E)", ylabel="P(vote)", ylim=(0, 1), xlim=(0, 30))
axes[0].legend(fontsize=8)

turnout_campaign = [CampaignAction("turnout_demo", "CAND_0001", 10, 15, 0.5, 0.25)]
turnout_cfg = replace(social_cfg, lambda_campaign=0.2)
gammas = [0.0, 0.03, 0.1, -0.03]
rows = []
for gamma in gammas:
    for seed in range(30):
        run = simulate(replace(turnout_cfg, gamma_mob=gamma), no_events, seed, turnout_campaign)
        rows.append({"gamma_mob": gamma, "seed": seed, "turnout_rate": run.turnout["turnout_rate"]})
turnout_runs = pd.DataFrame(rows)
stats = turnout_runs.groupby("gamma_mob", sort=False).turnout_rate.agg(
    mean="mean", p05=lambda s: s.quantile(.05), p95=lambda s: s.quantile(.95)).reindex(gammas)
labels = [("γ = 0\n(original)" if g == 0 else f"γ = {g:+g}\n" + ("mobilise" if g > 0 else "demobilise"))
          for g in gammas]
bar_colors = ["#94a3b8" if g == 0 else "#2563eb" if g > 0 else "#e11d48" for g in gammas]
axes[1].bar(labels, stats["mean"] * 100, color=bar_colors,
            yerr=[(stats["mean"] - stats.p05) * 100, (stats.p95 - stats["mean"]) * 100], capsize=4)
for i, (value, top) in enumerate(zip(stats["mean"] * 100, stats.p95 * 100)):
    axes[1].text(i, top + 0.4, f"{value:.1f}%", ha="center", va="bottom", fontsize=9)
lo = max(0, stats.p05.min() * 100 - 8)
axes[1].set(title="Turnout, C01, 25% campaign reach (30 seeds)", ylabel="Turnout (%)",
            ylim=(lo, stats.p95.max() * 100 + 6))
axes[1].grid(axis="x", visible=False)
save(fig, "slide15_turnout_boost")
plt.show()
print((stats * 100).round(2).to_string())

## 4. Public polls and strategic voting (slide 16)

Same run twice (C01, seed 42, smear campaign, `c01_news.csv` with a public poll on day 25):
`configs/news.json` (sincere) vs `configs/strategic.json` (`strategic_voting = 1`).
This reproduces the table on the slide and shows why the third-placed candidate's votes halve.

In [ ]:
news_events = ROOT / "event_scenarios" / "c01_news.csv"
news_data = load_constituency(DATA_DIR, CONSTITUENCY_ID, election_duration_days=DAYS, events_file=news_events)
smear = load_actions(ROOT / "campaign_actions" / "smear_c01.json", DAYS, set(candidate_ids))
configs = {name: SimulationConfig(**json.loads((ROOT / "configs" / f"{file}.json").read_text()))
           for name, file in [("Sincere", "news"), ("Strategic", "strategic")]}
runs = {name: simulate(cfg, news_data, SEED, smear) for name, cfg in configs.items()}

table = pd.DataFrame({name: run.candidate_results.set_index("candidate_id").votes
                      for name, run in runs.items()}).T
table.columns = [c.replace("CAND_000", "C") for c in table.columns]
poll = runs["Strategic"].diagnostics["public_polls"][0]
print(f"Public poll on day {poll['day']}: " +
      ", ".join(f"{c.replace('CAND_000', 'C')} {s:.1%}" for c, s in poll["shares"].items()),
      "| non-viable:", poll["non_viable"])
display(table)

fig, ax = plt.subplots(figsize=(6, 3.6), constrained_layout=True)
for name, style in [("Sincere", "--"), ("Strategic", "-")]:
    s = runs[name].daily_support.query("candidate_id == 'CAND_0002'")
    ax.plot(s.day, s.mean_support * 100, style, color=CAND_COLORS["CAND_0002"], lw=2, label=name)
ax.axvline(poll["day"], color="#475569", lw=1, ls=":")
ax.text(poll["day"] + 0.3, ax.get_ylim()[1] * 0.95, "public poll", fontsize=8, va="top")
ax.set(title="CAND_0002 mean support: sincere vs strategic", xlabel="Day",
       ylabel="Mean support (%)", xlim=(0, DAYS))
ax.legend(fontsize=8)
save(fig, "slide16_strategic_support")
plt.show()

## 5. Experiments and regional spillover (slides 17 and 18)

This runs the project's Step 5 pipeline (`election_sim/step5_cli.py`) exactly as the CLI does:

1. Scan all 21 constituencies with social influence + news and pick the closest race.
2. Run the scenario matrix in that constituency over `SEED_COUNT` matched seeds: frozen,
   social-only, ordinary / misleading / smear, social + news, everything together,
   scripted vs three agent strategies, and five targeting policies at equal 5% reach.
3. Run the region containing that constituency at spillover 0, 0.05, 0.1 and 0.2.

With `SEED_COUNT = 30` this is ~500 constituency runs plus 120 regional runs (roughly 10 minutes).
Results are cached in `results/presentation_step5_30seeds/`, so re-running the notebook is instant;
delete that folder to recompute.

In [ ]:
SEED_COUNT = 30
base_config = SimulationConfig(**json.loads((ROOT / "configs" / "baseline.json").read_text()))
region_data = load_region(DATA_DIR, days=DAYS)
TABLES = ("baseline_scan", "per_seed", "summary", "paired_effects", "spillover_per_seed", "spillover_summary")
CACHE = ROOT / "results" / f"presentation_step5_{SEED_COUNT}seeds"  # delete this folder to rerun
if (CACHE / "manifest.json").exists():
    step5 = {name: pd.read_csv(CACHE / f"{name}.csv") for name in TABLES}
    step5["manifest"] = json.loads((CACHE / "manifest.json").read_text())
    print(f"Loaded cached Step 5 results from {CACHE.relative_to(ROOT)}")
else:
    step5 = run_step5(base_config, region_data, seed=0, seed_count=SEED_COUNT)
    CACHE.mkdir(parents=True)
    for name in TABLES:
        step5[name].to_csv(CACHE / f"{name}.csv", index=False)
    (CACHE / "manifest.json").write_text(json.dumps(step5["manifest"], indent=2))
manifest = step5["manifest"]
CLOSEST = manifest["selected_constituency_id"]
print(f"Closest race: C{CLOSEST:02d} (top-two support gap {manifest['selected_top_two_support_margin']:.3f}); "
      f"region constituencies: {manifest['spillover_constituencies']}")

### Slide 17: seats and vote share vs spillover

Spillover scales cross-constituency social ties. At 0 the region reproduces the standalone
constituency results exactly (sanity check below).

In [ ]:
spill = step5["spillover_per_seed"].copy()
spill["spillover"] = spill.scenario.str.removeprefix("spillover_").astype(float)
values = sorted(spill.spillover.unique())
parties = sorted(spill.party_id.dropna().unique())

seats = (spill.loc[spill.winner].groupby(["spillover", "seed", "party_id"]).size()
         .unstack(fill_value=0).reindex(columns=parties, fill_value=0))
seat_mean = seats.groupby("spillover").mean()
votes = (spill.groupby(["spillover", "seed", "party_id"]).votes.sum().unstack(fill_value=0))
vote_share = votes.div(votes.sum(axis=1), axis=0).groupby("spillover")
vs_mean, vs_p05, vs_p95 = vote_share.mean(), vote_share.quantile(.05), vote_share.quantile(.95)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
x = np.arange(len(values))
bottom = np.zeros(len(values))
for party in parties:
    axes[0].bar(x, seat_mean[party], 0.6, bottom=bottom, color=PARTY_COLORS.get(party, None), label=party)
    for xi, b, h in zip(x, bottom, seat_mean[party]):
        if h > 0.3:
            axes[0].text(xi, b + h / 2, f"{h:.1f}", ha="center", va="center", color="white", fontsize=9)
    bottom += seat_mean[party].to_numpy()
axes[0].set_xticks(x, [f"{v:g}" for v in values])
axes[0].set(title=f"Mean seats won ({len(manifest['spillover_constituencies'])} constituencies, "
                  f"{SEED_COUNT} seeds)", xlabel="Spillover", ylabel="Seats")
axes[0].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
axes[0].grid(axis="x", visible=False)
share_by_seed = votes.div(votes.sum(axis=1), axis=0)
change = share_by_seed.sub(share_by_seed.xs(0.0, level="spillover"), level="seed") * 100  # pp vs spillover 0, same seed
change = change.groupby("spillover")
ch_mean, ch_p05, ch_p95 = change.mean(), change.quantile(.05), change.quantile(.95)
for party in parties:
    color = PARTY_COLORS.get(party, None)
    axes[1].plot(values, ch_mean[party], "o-", color=color, lw=2,
                 label=f"{party} ({vs_mean.loc[0.0, party]:.1%} at 0)")
    axes[1].fill_between(values, ch_p05[party], ch_p95[party], color=color, alpha=0.15)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set(title="Regional vote share vs spillover 0 (mean, 5–95% band)", xlabel="Spillover",
            ylabel="Change in vote share (pp)")
axes[1].set_xticks(values)
axes[1].legend(fontsize=8)
save(fig, "slide17_spillover")
plt.show()
print("Mean seats:\n", seat_mean.round(2).to_string())

In [ ]:
# Sanity check: spillover 0 equals the standalone constituency runs (same seed, same config).
from election_sim.experiments import _constituency_slice
check_seed = 0
regional0 = spill.query("spillover == 0 and seed == @check_seed").set_index("candidate_id").votes
standalone = pd.concat([
    simulate(replace(base_config, constituency_id=cid), _constituency_slice(region_data, cid),
             check_seed).candidate_results.set_index("candidate_id").votes
    for cid in manifest["spillover_constituencies"]])
print("Spillover 0 == standalone results:", regional0.sort_index().equals(standalone.sort_index()))

### Slide 18: vote-share change per scenario

Each bar is the change in vote share for the campaigning candidate (the first candidate ID in
the closest race), relative to its matched control on the same seed. Error bars are the
empirical 5–95% range over seeds. Controls follow `step5_cli.py`:
campaign types vs social-only, agent strategies vs the scripted equal-budget campaign,
targeting policies vs uniform targeting. Frozen, news and all-together are shown vs social-only.

In [ ]:
per_seed = step5["per_seed"]
lead = sorted(per_seed.candidate_id.unique())[0]

def paired_change(treatment, control):
    t = per_seed.query("scenario == @treatment").set_index(["seed", "candidate_id"]).vote_share
    c = per_seed.query("scenario == @control").set_index(["seed", "candidate_id"]).vote_share
    return (t - c).rename("change").reset_index().assign(comparison=treatment, control=control)

groups = [
    ("Model components\n(vs social-only)", [("frozen", "social_only"), ("social_plus_news", "social_only"),
                                         ("all_together", "social_only")]),
    ("Message type\n(vs social-only)", [("campaign_ordinary", "social_only"), ("campaign_misleading", "social_only"),
                                     ("campaign_smear", "social_only")]),
    ("Agents\n(vs scripted)", [("agents_greedy", "scripted_equal_budget"),
                               ("agents_go_negative", "scripted_equal_budget"),
                               ("agents_rebut", "scripted_equal_budget")]),
    ("Targeting\n(vs uniform)", [("target_group_students", "target_uniform"),
                                  ("target_high_susceptibility", "target_uniform"),
                                  ("target_high_influence", "target_uniform"),
                                  ("target_community", "target_uniform")]),
]
scenario_names = set(per_seed.scenario)
changes = pd.concat([paired_change(t, c) for _, pairs in groups for t, c in pairs
                     if t in scenario_names and c in scenario_names], ignore_index=True)
lead_changes = changes.query("candidate_id == @lead")
effect = lead_changes.groupby("comparison", sort=False).change.agg(
    mean="mean", p05=lambda s: s.quantile(.05), p95=lambda s: s.quantile(.95)) * 100

group_colors = ["#64748b", "#2563eb", "#9333ea", "#0d9488"]
fig, ax = plt.subplots(figsize=(11, 4.6), constrained_layout=True)
ypos, ticks, labels = 0, [], []
for (title, pairs), color in zip(groups, group_colors):
    names = [t for t, c in pairs if t in effect.index]
    for name in names:
        row = effect.loc[name]
        ax.errorbar(row["mean"], ypos, xerr=[[row["mean"] - row.p05], [row.p95 - row["mean"]]],
                    fmt="o", color=color, capsize=3, lw=1.5)
        ticks.append(ypos)
        labels.append(name.replace("campaign_", "").replace("agents_", "agent: ")
                      .replace("target_", "target: ").replace("_", " "))
        ypos += 1
    ax.text(1.01, ypos - len(names) / 2 - 0.5, title, transform=ax.get_yaxis_transform(),
            fontsize=8, va="center", color=color)
    ypos += 0.8
ax.axvline(0, color="k", lw=0.8)
ax.set_yticks(ticks, labels)
ax.invert_yaxis()
ax.set(title=f"Change in {lead} vote share, C{CLOSEST:02d} (mean and 5–95% over {SEED_COUNT} seeds)",
       xlabel="Vote-share change vs matched control (percentage points)")
ax.grid(axis="y", visible=False)
save(fig, "slide18_scenario_effects")
plt.show()

In [ ]:
# Table version for the slide: every scenario's vote share for each candidate (median, 5–95%).
summary = step5["summary"].copy()
summary["vote_share"] = summary.apply(
    lambda r: f"{r.vote_share_median:.1%} [{r.vote_share_p05:.1%}, {r.vote_share_p95:.1%}]", axis=1)
scenario_table = summary.pivot(index="scenario", columns="candidate_id", values="vote_share")
scenario_table = scenario_table.reindex(list(dict.fromkeys(per_seed.scenario)))
win_rate = summary.query("candidate_id == @lead").set_index("scenario").win_rate
scenario_table[f"{lead} win rate"] = win_rate.reindex(scenario_table.index).map("{:.0%}".format)
display(scenario_table)